In [1]:
import pandas as pd
import ast
import numpy as np

df = pd.read_csv('../data/RAW_recipes.csv')

df.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


In [2]:
cols_to_fix = ['tags', 'steps', 'ingredients', 'nutrition']

for col in cols_to_fix:
    df[col] = df[col].apply(ast.literal_eval)

print("Conversion complete!")

Conversion complete!


In [3]:
# สร้าง Column ใหม่แยกออกมา
df[['calories', 'total_fat', 'sugar', 'sodium', 'protein', 'sat_fat', 'carbs']] = pd.DataFrame(df['nutrition'].tolist(), index=df.index)

# ลบ Column เดิมทิ้งเพราะไม่ใช้แล้ว
df = df.drop(columns=['nutrition'])

# ดูผลลัพธ์
df[['name', 'calories', 'protein']].head()

,name,calories,protein
0,arriba baked winter squash mexican style,51.5,2.0
1,a bit different breakfast pizza,173.4,22.0
2,all in the kitchen chili,269.8,39.0
3,alouette potatoes,368.1,14.0
4,amish tomato ketchup for canning,352.9,3.0


In [4]:
#clean data
df_clean = df.copy()
#dropna ลบแถวที่มีค่าว่าง
df_clean = df_clean.dropna(subset=['name', 'steps', 'ingredients', 'calories'])
print(f"จำนวนเมนูอาหารนานาชาติทั้งหมด: {len(df_clean)} เมนู")


จำนวนเมนูอาหารนานาชาติทั้งหมด: 231636 เมนู


In [5]:
df.head()

,name,id,minutes,contributor_id,submitted,tags,n_steps,steps,description,ingredients,n_ingredients,calories,total_fat,sugar,sodium,protein,sat_fat,carbs
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, mai...",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,51.5,0.0,13.0,0.0,2.0,0.0,4.0
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, mai...",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,173.4,18.0,0.0,17.0,22.0,35.0,1.0
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish,...",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,269.8,22.0,32.0,48.0,39.0,27.0,5.0
3,alouette potatoes,59389,45,68585,2003-04-14,"[60-minutes-or-less, time-to-make, course, mai...",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,368.1,17.0,10.0,2.0,14.0,8.0,20.0
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"[weeknight, time-to-make, course, main-ingredi...",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,352.9,1.0,337.0,23.0,3.0,0.0,28.0


In [6]:
# Smart Categorization (Updated)
def categorize_recipe(row):
    name = str(row['name']).lower()
    steps = " ".join(row['steps']).lower()
    tags = " ".join(row['tags']).lower()
    full_text = name + " " + steps + " " + tags
    
    if any(x in full_text for x in ['dessert', 'cake', 'cookie', 'pie', 'chocolate', 'ice cream']):
        return 'Dessert' 
    elif any(x in full_text for x in ['breakfast', 'pancake', 'omelet', 'toast', 'egg']):
        return 'Breakfast' 
    elif any(x in full_text for x in ['pasta', 'spaghetti', 'noodle', 'macaroni']):
        return 'Pasta/Noodle' 
    elif any(x in full_text for x in ['salad', 'dressing']):
        return 'Salad' 
    elif any(x in full_text for x in ['soup', 'stew', 'chili', 'curry']):
        return 'Soup/Curry' 
    elif any(x in full_text for x in ['beverage', 'drink', 'smoothie', 'cocktail']):
        return 'Beverage' 
    elif any(x in full_text for x in ['bake', 'oven', 'roast', 'casserole']):
        return 'Baked/Roast' 
    elif any(x in full_text for x in ['fry', 'pan', 'skillet']):
        return 'Fried/Stir-Fry' 
    else:
        return 'Main Course' 

df_clean['category'] = df_clean.apply(categorize_recipe, axis=1)

print(df_clean['category'].value_counts())

category
Dessert           80846
Breakfast         45946
Baked/Roast       20898
Pasta/Noodle      18315
Soup/Curry        16033
Fried/Stir-Fry    15867
Main Course       14005
Salad             11766
Beverage           7960
Name: count, dtype: int64


In [7]:
df_clean.head()

,name,id,minutes,contributor_id,submitted,tags,n_steps,steps,description,ingredients,n_ingredients,calories,total_fat,sugar,sodium,protein,sat_fat,carbs,category
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, mai...",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,51.5,0.0,13.0,0.0,2.0,0.0,4.0,Dessert
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, mai...",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,173.4,18.0,0.0,17.0,22.0,35.0,1.0,Dessert
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish,...",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,269.8,22.0,32.0,48.0,39.0,27.0,5.0,Soup/Curry
3,alouette potatoes,59389,45,68585,2003-04-14,"[60-minutes-or-less, time-to-make, course, mai...",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,368.1,17.0,10.0,2.0,14.0,8.0,20.0,Breakfast
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"[weeknight, time-to-make, course, main-ingredi...",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,352.9,1.0,337.0,23.0,3.0,0.0,28.0,Main Course


In [8]:
#Prepare Ingredients for Search Engine
#ทำ String Aggregation List -> Flat String  เพื่อใช้ NLP เวลาทำ Recommendation
# df_clean['clean_ingredients'] = df_clean['ingredients'].apply(lambda x: ' '.join(x))
# df_clean[['name', 'category', 'clean_ingredients']].head()

In [9]:
df_clean.head()

,name,id,minutes,contributor_id,submitted,tags,n_steps,steps,description,ingredients,n_ingredients,calories,total_fat,sugar,sodium,protein,sat_fat,carbs,category
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"[60-minutes-or-less, time-to-make, course, mai...",11,"[make a choice and proceed with recipe, depend...",autumn is my favorite time of year to cook! th...,"[winter squash, mexican seasoning, mixed spice...",7,51.5,0.0,13.0,0.0,2.0,0.0,4.0,Dessert
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"[30-minutes-or-less, time-to-make, course, mai...",9,"[preheat oven to 425 degrees f, press dough in...",this recipe calls for the crust to be prebaked...,"[prepared pizza crust, sausage patty, eggs, mi...",6,173.4,18.0,0.0,17.0,22.0,35.0,1.0,Dessert
2,all in the kitchen chili,112140,130,196586,2005-02-25,"[time-to-make, course, preparation, main-dish,...",6,"[brown ground beef in large pot, add chopped o...",this modified version of 'mom's' chili was a h...,"[ground beef, yellow onions, diced tomatoes, t...",13,269.8,22.0,32.0,48.0,39.0,27.0,5.0,Soup/Curry
3,alouette potatoes,59389,45,68585,2003-04-14,"[60-minutes-or-less, time-to-make, course, mai...",11,[place potatoes in a large pot of lightly salt...,"this is a super easy, great tasting, make ahea...","[spreadable cheese with garlic and herbs, new ...",11,368.1,17.0,10.0,2.0,14.0,8.0,20.0,Breakfast
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"[weeknight, time-to-make, course, main-ingredi...",5,"[mix all ingredients& boil for 2 1 / 2 hours ,...",my dh's amish mother raised him on this recipe...,"[tomato juice, apple cider vinegar, sugar, sal...",8,352.9,1.0,337.0,23.0,3.0,0.0,28.0,Main Course


In [10]:
#save file cleaned_all_recipes
df_clean = df_clean.reset_index(drop=True)
df_clean.to_csv('../data/cleaned_all_recipes.csv', index=False)
df_clean.to_pickle('../data/cleaned_all_recipes.pkl')

print("save success!")

save success!
